In [0]:
# =============================================================================
# LANDING ZONE SYNTHETIC DATA GENERATOR (Databricks / PySpark)
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import datetime

# -------------------------------------------------------------------------
# CONFIGURATION
# -------------------------------------------------------------------------
LANDING_PATH = "/Volumes/main/default/landing_zone"
RUN_DATE = datetime.date.today()
RUN_DATE_STR = RUN_DATE.strftime("%Y%m%d")
RUN_TS_STR = datetime.datetime.now().strftime("%Y%m%d%H%M%S")

dbutils.widgets.dropdown("is_incremental", "false", ["true", "false"])
IS_DAILY_INCREMENTAL = dbutils.widgets.get("is_incremental") == "true"

# Base (One-Time Historical Bootstrap) volumes
NUM_CUSTOMERS = 100_354
NUM_ACCOUNTS = 150_052
NUM_MERCHANTS = 5_000
NUM_BILLERS = 289
NUM_INITIAL_TX_BACKFILL = 9_500_000  # Correctly configured for 1.5M transactions
BACKFILL_HISTORY_DAYS = 1095         # 3 full years backfill

# Daily incremental volumes
NEW_CUSTOMERS_PER_DAY = 284
NEW_ACCOUNTS_PER_DAY = 369
DAILY_TX_VOLUME = 6_159

# Mix of transaction types
TX_TYPE_WEIGHTS = {"purchase": 0.65, "bill_payment": 0.20, "transfer": 0.15}

# Commission table churn fraction
COMMISSION_DAILY_CHANGE_FRACTION = 0.12


def path(sub):
    return f"{LANDING_PATH}/{sub}"


def landing_exists(sub):
    try:
        dbutils.fs.ls(path(sub))
        return True
    except Exception:
        return False


# =============================================================================
# 1. CUSTOMERS (Parquet)
# =============================================================================
def generate_customers():
    print("\n--- Processing CUSTOMERS ---")
    if not IS_DAILY_INCREMENTAL:
        df = spark.range(1, NUM_CUSTOMERS + 1) \
            .withColumn("customer_id", F.concat(F.lit("CUST_"), F.lpad(F.col("id"), 8, "0"))) \
            .withColumn("first_name", F.elt((F.rand() * 5 + 1).cast("int"),
                        F.lit("John"), F.lit("Jane"), F.lit("Alex"), F.lit("Emily"), F.lit("Michael"))) \
            .withColumn("last_name", F.elt((F.rand() * 5 + 1).cast("int"),
                        F.lit("Smith"), F.lit("Doe"), F.lit("Johnson"), F.lit("Brown"), F.lit("Taylor"))) \
            .withColumn("primary_email", F.lower(F.concat(F.col("first_name"), F.lit("."), F.col("last_name"),
                        F.lit("@email.com")))) \
            .withColumn("secondary_email", F.when(F.rand() > 0.7,
                        F.lower(F.concat(F.col("first_name"), F.lit("_alt@email.com")))).otherwise(F.lit(None))) \
            .withColumn("phone_number", F.when(F.rand() > 0.1,
                        F.concat(F.lit("+1"), (F.rand() * 9_000_000_000 + 1_000_000_000).cast("long"))).otherwise(F.lit(None))) \
            .withColumn("country_code", F.elt((F.rand() * 4 + 1).cast("int"),
                        F.lit("US"), F.lit("CA"), F.lit("GB"), F.lit("DE"))) \
            .withColumn("credit_score", (F.rand() * 500 + 350).cast("int")) \
            .withColumn("kyc_status", F.when(F.rand() > 0.05, "VERIFIED").otherwise("PENDING")) \
            .withColumn("signup_date", F.date_sub(F.lit(RUN_DATE), (F.rand() * BACKFILL_HISTORY_DAYS).cast("int"))) \
            .withColumn("created_at", F.to_timestamp(F.col("signup_date"))) \
            .withColumn("updated_at", F.col("created_at")) \
            .drop("id")
        
        count = df.count()
        df.write.format("parquet").mode("overwrite").save(path("customers"))
        print(f"[CUSTOMERS] Initial historical backfill complete. Wrote {count:,} records.")
        return

    # Incremental: new signups today
    df_new = spark.range(1, NEW_CUSTOMERS_PER_DAY + 1) \
        .withColumn("customer_id", F.concat(F.lit("CUST_"), F.lit(RUN_TS_STR + "_"), F.lpad(F.col("id"), 6, "0"))) \
        .withColumn("first_name", F.elt((F.rand() * 5 + 1).cast("int"),
                    F.lit("John"), F.lit("Jane"), F.lit("Alex"), F.lit("Emily"), F.lit("Michael"))) \
        .withColumn("last_name", F.elt((F.rand() * 5 + 1).cast("int"),
                    F.lit("Smith"), F.lit("Doe"), F.lit("Johnson"), F.lit("Brown"), F.lit("Taylor"))) \
        .withColumn("primary_email", F.lower(F.concat(F.col("first_name"), F.lit("."), F.col("last_name"),
                    F.lit("@email.com")))) \
        .withColumn("secondary_email", F.when(F.rand() > 0.7,
                    F.lower(F.concat(F.col("first_name"), F.lit("_alt@email.com")))).otherwise(F.lit(None))) \
        .withColumn("phone_number", F.when(F.rand() > 0.1,
                    F.concat(F.lit("+1"), (F.rand() * 9_000_000_000 + 1_000_000_000).cast("long"))).otherwise(F.lit(None))) \
        .withColumn("country_code", F.elt((F.rand() * 4 + 1).cast("int"),
                    F.lit("US"), F.lit("CA"), F.lit("GB"), F.lit("DE"))) \
        .withColumn("credit_score", (F.rand() * 500 + 350).cast("int")) \
        .withColumn("kyc_status", F.when(F.rand() > 0.15, "VERIFIED").otherwise("PENDING")) \
        .withColumn("signup_date", F.lit(RUN_DATE)) \
        .withColumn("created_at", F.current_timestamp()) \
        .withColumn("updated_at", F.current_timestamp()) \
        .drop("id")
    
    new_count = df_new.count()

    # Incremental: updates to existing customers
    df_updates = None
    update_count = 0
    if landing_exists("customers"):
        df_existing = spark.read.parquet(path("customers"))
        df_updates = df_existing.sample(withReplacement=False, fraction=0.02) \
            .withColumn("credit_score", (F.rand() * 500 + 350).cast("int")) \
            .withColumn("secondary_email", F.when(F.rand() > 0.7,
                        F.lower(F.concat(F.col("first_name"), F.lit("_alt@email.com")))).otherwise(F.lit(None))) \
            .withColumn("kyc_status", F.when(F.rand() > 0.02, "VERIFIED").otherwise("PENDING")) \
            .withColumn("updated_at", F.current_timestamp())
        update_count = df_updates.count()

    df_out = df_new if df_updates is None else df_new.unionByName(df_updates)
    total_appended = df_out.count()
    df_out.write.format("parquet").mode("append").save(path("customers"))
    print(f"[CUSTOMERS] Incremental batch appended. New customers: {new_count:,} | Updated profile rows: {update_count:,} | Total added to stream: {total_appended:,}")


# =============================================================================
# 2. ACCOUNTS (Parquet)
# =============================================================================
def generate_accounts():
    print("\n--- Processing ACCOUNTS ---")
    if not IS_DAILY_INCREMENTAL:
        df = spark.range(1, NUM_ACCOUNTS + 1) \
            .withColumn("account_id", F.concat(F.lit("ACC_"), F.lpad(F.col("id"), 8, "0"))) \
            .withColumn("customer_id", F.concat(F.lit("CUST_"), F.lpad((F.rand() * NUM_CUSTOMERS + 1).cast("int"), 8, "0"))) \
            .withColumn("account_type", F.when(F.rand() > 0.3, "Checking").otherwise("Savings")) \
            .withColumn("account_status", F.elt((F.rand() * 4 + 1).cast("int"),
                        F.lit("Active"), F.lit("Active"), F.lit("Active"), F.lit("Dormant"))) \
            .withColumn("currency", F.lit("USD")) \
            .withColumn("created_at", F.to_timestamp(F.date_sub(F.lit(RUN_DATE), (F.rand() * BACKFILL_HISTORY_DAYS).cast("int")))) \
            .withColumn("updated_at", F.col("created_at")) \
            .drop("id")
        
        count = df.count()
        df.write.format("parquet").mode("overwrite").save(path("accounts"))
        print(f"[ACCOUNTS] Initial historical backfill complete. Wrote {count:,} records.")
        return

    # Incremental: new accounts
    df_customers = spark.read.parquet(path("accounts")).select("customer_id")
    df_new = spark.range(1, NEW_ACCOUNTS_PER_DAY + 1) \
        .withColumn("account_id", F.concat(F.lit("ACC_"), F.lit(RUN_TS_STR + "_"), F.lpad(F.col("id"), 6, "0"))) \
        .withColumn("account_type", F.when(F.rand() > 0.3, "Checking").otherwise("Savings")) \
        .withColumn("account_status", F.lit("Active")) \
        .withColumn("currency", F.lit("USD")) \
        .withColumn("created_at", F.current_timestamp()) \
        .withColumn("updated_at", F.current_timestamp()) \
        .drop("id")

    # Distributed assignment without Window single-partition bottleneck
    df_cust_sample = df_customers.sample(withReplacement=True, fraction=1.0).limit(NEW_ACCOUNTS_PER_DAY)
    df_new = df_new.withColumn("customer_id", F.concat(F.lit("CUST_"), F.lpad((F.rand() * NUM_CUSTOMERS + 1).cast("int"), 8, "0")))
    
    new_count = df_new.count()

    # Incremental: status changes on existing accounts
    df_status_updates = None
    update_count = 0
    if landing_exists("accounts"):
        df_existing = spark.read.parquet(path("accounts"))
        df_status_updates = df_existing.sample(withReplacement=False, fraction=0.01) \
            .withColumn("account_status", F.when(F.col("account_status") == "Active", "Dormant").otherwise("Active")) \
            .withColumn("updated_at", F.current_timestamp())
        update_count = df_status_updates.count()

    df_out = df_new if df_status_updates is None else df_new.unionByName(df_status_updates)
    total_appended = df_out.count()
    df_out.write.format("parquet").mode("append").save(path("accounts"))
    print(f"[ACCOUNTS] Incremental batch appended. New accounts: {new_count:,} | Account status changes: {update_count:,} | Total added to stream: {total_appended:,}")


# =============================================================================
# 3. MERCHANTS (CSV)
# =============================================================================
def generate_merchants():
    print("\n--- Processing MERCHANTS ---")
    if IS_DAILY_INCREMENTAL:
        print("[MERCHANTS] Incremental run - static reference table skipped.")
        return
    df = spark.range(1, NUM_MERCHANTS + 1) \
        .withColumn("merchant_id", F.concat(F.lit("MERCH_"), F.lpad(F.col("id"), 6, "0"))) \
        .withColumn("merchant_name", F.concat(F.lit("Store_"), F.col("id"))) \
        .withColumn("mcc_code", F.elt((F.rand() * 4 + 1).cast("int"),
                    F.lit("5411"), F.lit("5812"), F.lit("5732"), F.lit("5999"))) \
        .withColumn("country_code", F.elt((F.rand() * 4 + 1).cast("int"),
                    F.lit("US"), F.lit("CA"), F.lit("GB"), F.lit("DE"))) \
        .withColumn("onboarded_at", F.date_sub(F.lit(RUN_DATE), (F.rand() * BACKFILL_HISTORY_DAYS).cast("int"))) \
        .drop("id")
    
    count = df.count()
    df.write.format("csv").option("header", "true").mode("overwrite").save(path("merchants"))
    print(f"[MERCHANTS] Wrote {count:,} merchant dimension records.")


# =============================================================================
# 4. BILLERS (CSV)
# =============================================================================
def generate_billers():
    print("\n--- Processing BILLERS ---")
    if IS_DAILY_INCREMENTAL:
        print("[BILLERS] Incremental run - static reference table skipped.")
        return
    df = spark.range(1, NUM_BILLERS + 1) \
        .withColumn("biller_id", F.concat(F.lit("BILL_"), F.lpad(F.col("id"), 5, "0"))) \
        .withColumn("biller_name", F.concat(F.lit("Biller_"), F.col("id"))) \
        .withColumn("biller_category", F.elt((F.rand() * 6 + 1).cast("int"),
                    F.lit("Electricity"), F.lit("Water"), F.lit("Internet"),
                    F.lit("Mobile_Topup"), F.lit("Insurance"), F.lit("Education"))) \
        .withColumn("country_code", F.elt((F.rand() * 4 + 1).cast("int"),
                    F.lit("US"), F.lit("CA"), F.lit("GB"), F.lit("DE"))) \
        .drop("id")
    
    count = df.count()
    df.write.format("csv").option("header", "true").mode("overwrite").save(path("billers"))
    print(f"[BILLERS] Wrote {count:,} biller dimension records.")


# =============================================================================
# 5. COMMISSION RULES (Parquet)
# =============================================================================
COMMISSION_SCOPES = [
    ("purchase", "5411"), ("purchase", "5812"), ("purchase", "5732"), ("purchase", "5999"),
    ("bill_payment", "Electricity"), ("bill_payment", "Water"), ("bill_payment", "Internet"),
    ("bill_payment", "Mobile_Topup"), ("bill_payment", "Insurance"), ("bill_payment", "Education"),
    ("transfer", "internal"),
]


def generate_commission_rules():
    print("\n--- Processing COMMISSION RULES ---")
    if not IS_DAILY_INCREMENTAL:
        import random
        rows = []
        for tx_type, scope_value in COMMISSION_SCOPES:
            rows.append((
                f"RULE_{tx_type}_{scope_value}",
                tx_type,
                scope_value,
                "percentage" if tx_type != "transfer" else "flat",
                float(round(random.uniform(0.010, 0.035), 4)) if tx_type != "transfer" else 0.50,
            ))
        schema = StructType([
            StructField("commission_rule_id", StringType()),
            StructField("scope_type", StringType()),
            StructField("scope_value", StringType()),
            StructField("commission_type", StringType()),
            StructField("commission_value", DoubleType()),
        ])
        df = spark.createDataFrame(rows, schema) \
            .withColumn("updated_at", F.current_timestamp())
        
        count = df.count()
        df.write.format("parquet").mode("overwrite").save(path("commission_rules"))
        print(f"[COMMISSION RULES] Initial setup complete. Wrote {count:,} rules.")
        return

    df_existing = spark.read.parquet(path("commission_rules"))
    df_out = df_existing \
        .withColumn("_reprice", F.rand() < COMMISSION_DAILY_CHANGE_FRACTION) \
        .withColumn("commission_value", F.when(F.col("_reprice"),
                    F.round(F.col("commission_value") * (F.lit(1) + (F.rand() * 0.4 - 0.2)), 4))
                    .otherwise(F.col("commission_value"))) \
        .withColumn("updated_at", F.when(F.col("_reprice"), F.current_timestamp())
                    .otherwise(F.col("updated_at")))
    
    repriced_count = df_out.filter(F.col("_reprice") == True).count()
    total_count = df_out.count()
    
    df_out = df_out.drop("_reprice")
    df_out.write.format("parquet").mode("overwrite").save(path("commission_rules"))
    print(f"[COMMISSION RULES] Overwrote state table. Total active rules: {total_count:,} | Rules re-priced today: {repriced_count:,}")


# =============================================================================
# 6. TRANSACTIONS (JSON)
# =============================================================================
def _new_transactions(df_accounts, df_merchants, df_billers, n_rows, id_prefix_seed, spread_over_history=False):
    df = spark.range(1, n_rows + 1) \
        .withColumn("transaction_id", F.concat(F.lit("TX_"), F.lit(id_prefix_seed + "_"), F.lpad(F.col("id"), 8, "0"))) \
        .withColumn("_r", F.rand())

    df = df.withColumn(
        "transaction_type",
        F.when(F.col("_r") < TX_TYPE_WEIGHTS["purchase"], "purchase")
         .when(F.col("_r") < TX_TYPE_WEIGHTS["purchase"] + TX_TYPE_WEIGHTS["bill_payment"], "bill_payment")
         .otherwise("transfer")
    ).drop("_r").drop("id")

    n_merchants = df_merchants.count()
    n_billers = df_billers.count()

    # Fast distributed generation for foreign keys
    df = df.withColumn("account_id", F.concat(F.lit("ACC_"), F.lpad((F.rand() * NUM_ACCOUNTS + 1).cast("int"), 8, "0")))
    df = df.withColumn("destination_account_id", F.concat(F.lit("ACC_"), F.lpad((F.rand() * NUM_ACCOUNTS + 1).cast("int"), 8, "0")))

    df = df.withColumn("destination_account_id",
        F.when(F.col("transaction_type") == "transfer", F.col("destination_account_id")).otherwise(F.lit(None)))

    df = df \
        .withColumn("merchant_id", F.when(F.col("transaction_type") == "purchase",
                    F.concat(F.lit("MERCH_"), F.lpad((F.rand() * n_merchants + 1).cast("int"), 6, "0"))).otherwise(F.lit(None))) \
        .withColumn("biller_id", F.when(F.col("transaction_type") == "bill_payment",
                    F.concat(F.lit("BILL_"), F.lpad((F.rand() * n_billers + 1).cast("int"), 5, "0"))).otherwise(F.lit(None))) \
        .withColumn("biller_reference_number", F.when(F.col("transaction_type") == "bill_payment",
                    F.concat(F.lit("REF"), (F.rand() * 900000000 + 100000000).cast("long"))).otherwise(F.lit(None)))

    df = df.withColumn("amount",
        F.when(F.col("transaction_type") == "purchase", F.round(F.rand() * 1200 + 1.50, 2))
         .when(F.col("transaction_type") == "bill_payment", F.round(F.rand() * 300 + 10.00, 2))
         .otherwise(F.round(F.rand() * 5000 + 5.00, 2))
    )

    df = df.withColumn("channel",
        F.when(F.col("transaction_type") == "purchase",
               F.elt((F.rand() * 4 + 1).cast("int"), F.lit("Web"), F.lit("Mobile"), F.lit("POS"), F.lit("ATM")))
         .otherwise(F.elt((F.rand() * 2 + 1).cast("int"), F.lit("Web"), F.lit("Mobile")))
    )

    df = df.withColumn("card_number",
        F.when((F.col("transaction_type") == "purchase") & (F.col("channel").isin("Web", "POS")),
               F.concat(F.lit("4532xxxxxx"), F.lpad((F.rand() * 9999).cast("int"), 4, "0"))
        ).otherwise(F.lit(None)))

    df = df.withColumn("ip_address",
        F.when(F.col("channel").isin("Web", "Mobile"),
               F.concat((F.rand() * 200 + 1).cast("int"), F.lit("."), (F.rand() * 200 + 1).cast("int"), F.lit(".1.10"))
        ).otherwise(F.lit(None)))

    df = df.withColumn("currency", F.lit("USD"))

    if spread_over_history:
        df = df.withColumn("_days_ago", (F.rand() * BACKFILL_HISTORY_DAYS).cast("int"))
    else:
        df = df.withColumn("_days_ago", F.lit(0))

    df = df.withColumn("_seconds_in_day", (F.rand() * 86400).cast("int"))
    df = df.withColumn("_status_r", F.rand())

    base_unix = F.unix_timestamp(F.lit(str(RUN_DATE)), "yyyy-MM-dd")
    df = df.withColumn("created_at", F.from_unixtime(
        base_unix - (F.col("_days_ago") * 86400) + F.col("_seconds_in_day")).cast("timestamp"))

    df = df.withColumn("status",
        F.when(F.col("_days_ago") == 0,
               F.when(F.col("_status_r") > 0.20, "SETTLED").otherwise("PENDING"))
         .otherwise(
             F.when(F.col("_status_r") < 0.90, "SETTLED")
              .when(F.col("_status_r") < 0.96, "DECLINED")
              .otherwise("CANCELLED")
         ))

    df = df.withColumn("updated_at", F.col("created_at")) \
        .drop("_days_ago", "_seconds_in_day", "_status_r")

    return df


def generate_transactions():
    print("\n--- Processing TRANSACTIONS ---")
    df_accounts = spark.read.parquet(path("accounts"))
    df_merchants = spark.read.csv(path("merchants"), header=True)
    df_billers = spark.read.csv(path("billers"), header=True)

    if not IS_DAILY_INCREMENTAL:
        # Fixed: Explicitly passing NUM_INITIAL_TX_BACKFILL (1,500,000)
        df_tx = _new_transactions(df_accounts, df_merchants, df_billers,
                                   NUM_INITIAL_TX_BACKFILL, RUN_TS_STR, spread_over_history=True)
        count = df_tx.count()
        df_tx.write.format("json").mode("overwrite").save(path("transactions"))
        print(f"[TRANSACTIONS] Initial historical backfill complete. Wrote {count:,} raw transactions over 3 years.")
        return

    df_new = _new_transactions(df_accounts, df_merchants, df_billers, DAILY_TX_VOLUME, RUN_TS_STR)
    new_count = df_new.count()

    df_pending_updates = None
    pending_count = 0
    if landing_exists("transactions"):
        df_all = spark.read.json(path("transactions"))
        df_pending = df_all.filter(F.col("status") == "PENDING")
        if df_pending.take(1):
            df_pending_updates = df_pending \
                .withColumn("status", F.elt((F.rand() * 3 + 1).cast("int"),
                            F.lit("SETTLED"), F.lit("DECLINED"), F.lit("CANCELLED"))) \
                .withColumn("updated_at", F.current_timestamp())
            pending_count = df_pending_updates.count()

    df_out = df_new if df_pending_updates is None else df_new.unionByName(df_pending_updates)
    total_appended = df_out.count()
    df_out.write.format("json").mode("append").save(path("transactions"))
    print(f"[TRANSACTIONS] Incremental batch appended. New raw events today: {new_count:,} | PENDING transactions resolved: {pending_count:,} | Total added to stream: {total_appended:,}")


# =============================================================================
# RUN ENGINE
# =============================================================================
if __name__ == "__main__":
    print(f"=================================================================")
    print(f" STARTING LANDING ZONE GENERATOR | IS_INCREMENTAL={IS_DAILY_INCREMENTAL}")
    print(f" DATE: {RUN_DATE} | PATH: {LANDING_PATH}")
    print(f"=================================================================")
    
    generate_customers()
    generate_accounts()
    generate_merchants()
    generate_billers()
    generate_commission_rules()
    generate_transactions()
    
    print(f"\n=================================================================")
    print(f" SUCCESS: Landing zone refreshed at {LANDING_PATH}")
    print(f"=================================================================")

In [0]:
# %sql
# DROP TABLE IF EXISTS main.bronze.raw_transactions;
# DROP TABLE IF EXISTS main.bronze.raw_accounts;
# DROP TABLE IF EXISTS main.bronze.raw_customers;
# DROP TABLE IF EXISTS main.bronze.raw_merchants;
# DROP TABLE IF EXISTS main.bronze.raw_billers;
# DROP TABLE IF EXISTS main.bronze.raw_commission_rules;

In [0]:
# %sql
# -- =============================================================================
# -- FULL DATA ENVIRONMENT TEARDOWN (Databricks %sql Cell)
# -- =============================================================================

# -- 1. BRONZE LAYER TABLES
# DROP TABLE IF EXISTS main.bronze.raw_transactions;
# DROP TABLE IF EXISTS main.bronze.raw_accounts;
# DROP TABLE IF EXISTS main.bronze.raw_customers;
# DROP TABLE IF EXISTS main.bronze.raw_merchants;
# DROP TABLE IF EXISTS main.bronze.raw_billers;
# DROP TABLE IF EXISTS main.bronze.raw_commission_rules;

# -- 2. STAGING VIEWS
# DROP VIEW IF EXISTS stg_accounts;
# DROP VIEW IF EXISTS stg_billers;
# DROP VIEW IF EXISTS stg_customers;
# DROP VIEW IF EXISTS stg_merchants;
# DROP VIEW IF EXISTS stg_transactions;

# -- 3. MEDALLION TABLES & VIEWS
# DROP TABLE IF EXISTS silver;
# DROP TABLE IF EXISTS gold;
# DROP TABLE IF EXISTS bronze;

# -- 4. METADATA TABLES (If created manually)
# DROP TABLE IF EXISTS table_constraints;
# DROP TABLE IF EXISTS table_privileges;
# DROP TABLE IF EXISTS table_tags;
# DROP TABLE IF EXISTS tables;
# DROP TABLE IF EXISTS views;
# DROP TABLE IF EXISTS volume_privileges;
# DROP TABLE IF EXISTS volume_tags;
# DROP TABLE IF EXISTS volumes;
# DROP TABLE IF EXISTS schemata;

# -- 5. DROP SCHEMAS / DATABASES (CASCADE deletes all contained objects)
# DROP SCHEMA IF EXISTS bronze CASCADE;
# DROP SCHEMA IF EXISTS silver CASCADE;
# DROP SCHEMA IF EXISTS gold CASCADE;
# DROP SCHEMA IF EXISTS assignmentscala CASCADE;
# DROP SCHEMA IF EXISTS rowan_depi CASCADE;

# -- -- 6. DROP CATALOGS
# -- DROP CATALOG IF EXISTS depicatalog CASCADE;